## RECAP – Sieci Rekurencyjne (RNN)

Sieci rekurencyjne (RNN) są przeznaczone do pracy z danymi sekwencyjnymi (np. tekst, szeregi czasowe). Ich kluczową cechą jest pamięć – model wykorzystuje informacje z poprzednich kroków sekwencji.

Najważniejsze właściwości:
- przetwarzanie danych krok po kroku (sekwencyjnie)
- wykorzystanie stanu ukrytego (hidden state) jako pamięci
- możliwość modelowania kontekstu

Problem klasycznych RNN:
- trudność w uczeniu długich zależności (vanishing gradient)

Typy komórek:
- SimpleRNN – prosty model, działa dobrze dla krótkich sekwencji
- LSTM – lepsze modelowanie długiego kontekstu dzięki mechanizmowi bramek
- GRU – uproszczona wersja LSTM, szybsza i często równie skuteczna

## W tym notebooku ważna będzie zmiana środowiska na wspierające GPU.

In [ ]:
!pip install -U transformers tensorflow

# Praca z tekstem i modele przeznaczone do takich zadań

In [ ]:
import sys
import numpy as np
import tensorflow as tf
import copy
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.datasets import imdb
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, Flatten, Dense, Dropout, GRU, GlobalAveragePooling1D
from transformers import pipeline
from sklearn.metrics import accuracy_score
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.utils import to_categorical

## Stemming i lematyzacja – przetwarzanie tekstu

**Stemming** i **lematyzacja** to techniki upraszczania słów do ich form podstawowych.

- Stemming usuwa końcówki fleksyjne (np. running → run)  
  jest szybki, ale mniej precyzyjny  

- Lematyzacja wykorzystuje słowniki językowe (np. better → good)  
  jest dokładniejsza, ale wolniejsza  

Obie metody pomagają ujednolicić tekst i mogą poprawić wyniki modeli NLP.


#### Ciekawostka – język polski

W języku polskim stemming i lematyzacja są znacznie trudniejsze niż w angielskim.

- polski jest językiem silnie fleksyjnym (wiele form jednego słowa)
- np. „kot” → kota, kotem, kotu, kotów, kotami  

Stemming często daje nienaturalne lub błędne formy, dlatego:
- w praktyce częściej stosuje się lematyzację  
- wymaga ona jednak zaawansowanych narzędzi (np. Morfeusz, spaCy pl)

W NLP dla języka polskiego preprocessing ma więc większe znaczenie niż w angielskim.

In [ ]:
import nltk
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Pobieranie potrzebnych pakietów
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('punkt_tab')

In [ ]:
# Przykładowy tekst
text = "We love programming on a piece of paper."

# Tokenizacja wyrazów
words = word_tokenize(text)

In [ ]:
# Inicjalizacja Stemmera i Lemmatizera
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

# Stemming wyrazów
stemmed_words = [stemmer.stem(word) for word in words]

# Lematyzacja wyrazów
lemmatized_words = [lemmatizer.lemmatize(word) for word in words]

In [ ]:
print("Original Words:    ", words)
print("Stemmed Words:     ", stemmed_words)
print("Lemmatized Words:  ", lemmatized_words)

### **Zadanie na zajęciach 1**

Przetestuj działanie stemmingu oraz lematyzacji na własnym przykładzie.

1. Wymyśl krótkie zdanie (np. w języku angielskim lub polskim).
2. Zastosuj stemming oraz lematyzację do każdego słowa w zdaniu.
3. Porównaj wyniki obu metod:
   - które słowa zostały zmienione?
   - gdzie pojawiają się różnice?
4. Zastanów się:
   - która metoda daje bardziej naturalne wyniki?
   - w jakich zadaniach NLP dana metoda może być bardziej użyteczna?

### Reprezentacja tekstu w sieci neuronowej

Aby model mógł przetwarzać tekst, musi on zostać zamieniony na formę numeryczną.


### Tokenizacja – podział tekstu na wyrazy

Tokenizacja to proces dzielenia tekstu na mniejsze elementy (tokeny), najczęściej słowa.

Przykład:
"To jest zdanie." → ["To", "jest", "zdanie"]

Dlaczego tokenizacja jest ważna:
- jest pierwszym krokiem w przetwarzaniu tekstu
- umożliwia dalsze operacje (np. stemming, lematyzacja, embeddingi)

Uwagi:
- tokenem nie zawsze musi być całe słowo (np. subword tokenization w nowoczesnych modelach)
- sposób tokenizacji wpływa na jakość modelu

In [ ]:
# Tokenizacja tekstu
text = "Never have I ever complained about SSN classes"
tokens = text.split(" ")
tokens

### Kodowanie słów jako liczby

Aby model mógł pracować na tekście, każde słowo musi zostać zamienione na liczbę.

Proces:
- budujemy słownik (vocabulary) wszystkich unikalnych słów w zbiorze danych
- każdemu słowu przypisujemy unikalny indeks (liczbę)

Przykład:
["kot", "pies", "kot"] → {"kot": 1, "pies": 2} → [1, 2, 1]

Uwagi:
- model operuje na indeksach, nie na surowym tekście
- wielkość słownika wpływa na złożoność modelu

In [ ]:
# Proste enkodowanie numeryczne
token_to_index = {word: idx + 1 for idx, word in enumerate(tokens)}
encoded_text = list(token_to_index.values())
encoded_text

### Wyrównywanie długości sekwencji (padding)

Sieci neuronowe wymagają stałego wymiaru wektora cech (np. embeddingu, tokenu) dla każdego elementu sekwencji. Natomiast podczas trenowania w batchach konieczne jest ujednolicenie długości sekwencji, dlatego stosuje się padding lub przycinanie.

### **Zadanie na zajęciach 2**

1. Uzupełnij parametr `maxlen` w funkcji `pad_sequences`:
   - ustaw go na długość pierwszej sekwencji z listy `sequences`

2. Uruchom kod i przeanalizuj wynik.

3. Odpowiedz na pytania:
   - co dzieje się z krótszymi sekwencjami?
   - co dzieje się z dłuższymi sekwencjami?
   - gdzie dodawane są wartości (na początku czy na końcu)?
   - jaka wartość jest używana do paddingu?

Wskazówka:
- padding dodaje brakujące wartości (najczęściej 0)
- przycinanie usuwa nadmiarowe elementy sekwencji

In [ ]:
# Padding - wypełnianie zaenkodowanych sekwencji żeby były jednej długości
from tensorflow.keras.preprocessing.sequence import pad_sequences

sequences = [[1, 2, 3, 4, 5, 6, 7, 8],[1, 1, 1, 2, 3, 4, 5, 6, 7, 8],  [1, 2, 3, 4, 5, 6]]
padded_sequences = pad_sequences(sequences, maxlen=<PUT_VALUE_HERE>, padding="post")
print(padded_sequences)


### Bag of Words (BoW)

Model Bag of Words (BoW) to prosta metoda reprezentacji tekstu w formie numerycznej.

Założenia:
- tworzymy słownik wszystkich unikalnych słów w korpusie
- każdy dokument reprezentujemy jako wektor długości słownika
- wartości wektora oznaczają liczbę wystąpień danego słowa

Przykład:
Dokument: "kot pies kot"  
Słownik: {"kot": 0, "pies": 1}  
Wektor: [2, 1]

Ograniczenia:
- brak informacji o kolejności słów
- brak uchwycenia kontekstu

Rozszerzenie:
- TF-IDF – uwzględnia ważność słów w dokumentach, redukując wpływ bardzo częstych słów

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# Przykładowe zdania
corpus = [
    "I love AGH",
    "The longer I think about it the less I understand it",
    "It is better to walk slowly in a good diirection than run around a false hope"
]

# Inicjalizacja COunt Vectorizera
vectorizer = CountVectorizer(token_pattern=r"(?u)\b\w+\b")

# Fit transform
X = vectorizer.fit_transform(corpus)

print("Nazwy cech", vectorizer.get_feature_names_out())
print("Reprezentacja BoW:\n", X.toarray())


### **Zadanie na zajęciach 3**

Przetestuj reprezentację Bag of Words (BoW) dla zdania:

"The longer I study at AGH the less I understand why, but I trust the process"

1. Zamień zdanie na reprezentację BoW.
2. Sprawdź, jakie wartości przyjmuje wektor.
3. Zinterpretuj wynik:
   - które słowa mają największe wartości i dlaczego?
   - jak częstość występowania słów wpływa na wynik?
4. Zastanów się:
   - czy model uwzględnia kolejność słów?
   - czy na podstawie wektora można zrozumieć znaczenie zdania?

### TF-IDF  (EXTRA)

TF-IDF (Term Frequency–Inverse Document Frequency) to metoda ważenia słów w dokumentach.

Intuicja:
- słowa częste w jednym dokumencie są ważne
- słowa częste we wszystkich dokumentach są mniej istotne

Składniki:

- TF (Term Frequency)  
  częstość słowa w dokumencie = liczba wystąpień słowa / liczba wszystkich słów  

- IDF (Inverse Document Frequency)  
  rzadkość słowa w zbiorze dokumentów  = log(N / df), gdzie:
  - N – liczba dokumentów  
  - df – liczba dokumentów zawierających dane słowo  

Końcowa wartość:
- TF-IDF = TF × IDF

Cel:
- wyodrębnienie słów najlepiej opisujących dokument  
- zmniejszenie wpływu słów bardzo częstych (mało informacyjnych)

### Word embeddings

Tradycyjne metody (kodowanie indeksowe, Bag of Words, TF-IDF) reprezentują tekst numerycznie, ale słabo uwzględniają znaczenie słów i ich kontekst. Słowa o podobnym znaczeniu są traktowane jako niezależne.

**Word embeddings** to reprezentacje wektorowe słów, które kodują ich znaczenie w przestrzeni numerycznej.

Cechy:
- gęste wektory o stałej długości  
- podobne słowa mają podobne reprezentacje  
- uwzględniają kontekst występowania słów  

Intuicja:
- słowa pojawiające się w podobnym kontekście mają podobne znaczenie  
- ich wektory są blisko siebie w przestrzeni  

Uczenie:
- modele uczą się reprezentacji na podstawie dużych zbiorów tekstów  
- przykłady: Word2Vec, modele oparte na transformerach  

Wniosek:
- embeddings pozwalają uchwycić relacje semantyczne i syntaktyczne między słowami

### Architektury Word Embedding (Word2Vec)

Dwa podstawowe podejścia do uczenia embeddingów:

#### CBOW (Continuous Bag of Words)

Model przewiduje słowo na podstawie jego kontekstu (otaczających słów).

Przykład:
- Input: ["The", "longer", "I", "about", "it"]  
- Label: "think"

Cechy:
- szybszy w trenowaniu  
- dobrze działa dla częstych słów  

---

#### Skip-gram

Model przewiduje kontekst na podstawie pojedynczego słowa.

Przykład:
- Input: "think"  
- Label: ["The", "longer", "I", "about", "it"]

Cechy:
- lepiej radzi sobie z rzadkimi słowami  
- wolniejszy, ale często dokładniejszy  

---

Poniżej znajduje się przykładowa implementacja modelu Word2Vec w Keras.  
Dane wejściowe są odpowiednio przygotowane (augmentowane), aby dopasować je do architektury modelu.

In [ ]:
# Defninicja danych testowych
corpus = [
    "I love AGH",
    "The longer I think about it the less I understand it",
    "It is better to walk slowly in a good direction than run around a false hope"
]

input_values = [
    "I love",
    "The longer I think about it the I understand it",
    "It is better to walk slowly in a good direction than run around a hope"
]

label = [
    "AGH",
    "less",
    "false"
]

# Przygotowanie danych do modelu

# Tokenizacja
tokenizer = Tokenizer()
tokenizer.fit_on_texts(corpus)

# Definicja rozmiaru wektora wejsciwego - ilość słów w słowniku
vocab_size = len(tokenizer.word_index) + 1

# Przygotowanie sekwencji
train_sequences = tokenizer.texts_to_sequences(input_values)
train_padded_sequences = np.array(pad_sequences(train_sequences, maxlen= max(len(seq) for seq in train_sequences), padding='post'))

# Przygotowanie sekwencji
label_sequences = np.array(tokenizer.texts_to_sequences(label))


In [ ]:
print(train_padded_sequences)
print(label_sequences)

In [ ]:
# Definicja modelu WordEmbedding
embedding_dim = 8  # Define embedding size
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=train_padded_sequences.shape[1]),
    # tworzy macierz embeddingów o rozmiarze (vocab_size × embedding_dim), gdzie
    # embedding_dim to liczba cech (wymiar wektora) reprezentującego każde słowo
    GlobalAveragePooling1D(),
    Dense(8, activation='relu'),  
    Dense(vocab_size, activation='softmax')
])

# Kompilacja modelu
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')

In [ ]:
# Trening modelu na zaugmentowanych danych
model.fit(train_padded_sequences, label_sequences.flatten(), epochs=10, verbose=1)


In [ ]:
# Ekstrakcja embeddingów dla poszczególnych słow modelu
embedding_layer = model.layers[0]
embedding_matrix = embedding_layer.get_weights()[0]

In [ ]:
embedding_matrix.shape

In [ ]:
# Wyciągnięcie embeddingów dla poszczególnych słow
word_embeddings = {word: embedding_matrix[idx] for word, idx in tokenizer.word_index.items()}

# Przykład użycia
word = "love"
if word in word_embeddings:
    print(f"Vector for word '{word}':")
    print(word_embeddings[word])
else:
    print(f"Word '{word}' not in vocabulary")

### **Zadanie na zajęciach 4**

Przetestuj działanie modelu embeddingów (Word2Vec) na własnym przykładzie.

1. Zdefiniuj własne zdanie.
2. Wykonaj preprocessing:
   - tokenizację
   - zamianę słów na indeksy
   - padding sekwencji
3. Podaj przygotowane dane do modelu i wykonaj predykcję.

4. Przeanalizuj wynik:
   - jaki kształt (shape) ma wynik predykcji?
   - od czego zależy jego rozmiar?

Wskazówka:
- rozmiar wyjścia zależy m.in. od:
  - liczby słów (długości sekwencji)
  - rozmiaru embeddingu (embedding_dim)

### Analiza sentymentu z wykorzystaniem LSTM (IMDb)

W tej części wykorzystamy sieć LSTM do analizy sentymentu recenzji filmów ze zbioru IMDb.

Cel:
- klasyfikacja tekstu (pozytywna / negatywna opinia)
- wykorzystanie zdolności LSTM do modelowania sekwencji

Uwaga:
- trenowanie modeli sekwencyjnych może być czasochłonne

**!!!Wskazówka!!!**:
- warto uruchomić notatnik w środowisku z GPU (np. Google Colab z GPU)

Przygotowanie danych do treningu

In [ ]:
# Load IMDB dataset

# maksymalna liczba słów w słowniku (najczęściej występujących)
# ignorujemy rzadsze słowa, aby zmniejszyć złożoność modelu
max_features = 10000

# maksymalna długość sekwencji (liczba słów w jednej recenzji)
# wszystkie sekwencje będą przycięte lub uzupełnione do tej długości
max_len = 200

# wczytanie danych IMDb:
# x_train, x_test → listy zakodowanych recenzji (każde słowo jako liczba)
# y_train, y_test → etykiety (0 = negatywna, 1 = pozytywna)
# num_words=max_features ogranicza słownik do 10000 najczęstszych słów
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=max_features)

# padding sekwencji treningowych:
# - jeśli sekwencja jest krótsza niż max_len → dodawane są zera
# - jeśli jest dłuższa → zostaje przycięta
# dzięki temu wszystkie wejścia mają ten sam rozmiar
x_train = pad_sequences(x_train, maxlen=max_len)

# padding sekwencji testowych (analogicznie jak dla treningowych)
x_test = pad_sequences(x_test, maxlen=max_len)

### **Zadanie na zajęciach 5**

Zdefiniuj model sieci neuronowej wykorzystujący warstwy Embedding oraz SimpleRNN (np. w Keras Sequential).

Sprawdź, jak wygląda wejście do modelu oraz jaki kształt ma jego wyjście.

Cel:
- zrozumienie, jak tekst (indeksy słów) jest przekształcany w wektory i przetwarzany przez RNN

In [ ]:
rnn_model = Sequential([
    <PLACEHOLDER_FOR_EMBEDDING_LAYER>,
    <PLACEHOLDER_FOR_SIMPLERNN_LAYER>,
    Dense(1, activation='sigmoid')
])

rnn_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])


In [ ]:
# Trening i ewaluacja modelu SimpleRNN
rnn_model.fit(x_train, y_train, epochs=3, batch_size=64, validation_data=(x_test, y_test))
rnn_acc = rnn_model.evaluate(x_test, y_test, verbose=0)[1]

### **Zadanie na zajęciach 6**

Zdefiniuj model LSTM do predykcji sentymentu dla zbioru IMDb i wytrenuj go analogicznie do modelu SimpleRNN.

Po treningu sprawdź wartość metryki accuracy na zbiorze treningowym.

Cel:
- porównanie działania LSTM z prostym RNN w zadaniu klasyfikacji tekstu

In [ ]:
lstm_model = Sequential

In [ ]:
# Trening i ewaluacja modelu LSTM
lstm_model.fit(x_train, y_train, epochs=3, batch_size=64, validation_data=(x_test, y_test))
lstm_acc = lstm_model.evaluate(x_test, y_test, verbose=0)[1]

Przygotowanie tekstu z tokenów dla porównania dokładności klasyfikacji RNN z pretrenowanym transformerem.

In [ ]:
# Pobranie i stworzenie mapy dla zbioru danych imbd
word_index = imdb.get_word_index()

# Odwrocenie word_index na tekst
index_word = {v + 3: k for k, v in word_index.items()}  # Offset by 3 (IMDB special tokens)
index_word[0], index_word[1], index_word[2], index_word[3] = "<PAD>", "<START>", "<UNK>", "<UNUSED>"

In [ ]:
def decode_review(encoded_review):
    return " ".join([index_word.get(i, "?") for i in encoded_review])

In [ ]:
# Dekodowanie tokenow
x_test_texts = [decode_review(seq) for seq in x_test]

Przykładowy zdekodowany tekst

In [ ]:
x_test_texts[1]

Ładowanie transformera z biblioteki hugging face

In [ ]:
sentiment_pipeline = pipeline("sentiment-analysis")

In [ ]:
# Przykład predykcji
sentiment_pipeline(x_test_texts[1])

In [ ]:
# Mapa do etykiet dla transformera
map_label = {"NEGATIVE": 0, "POSITIVE": 1}

Predykcje z użyciem transformera.

**NOTE**: Predykcje mogą chwile potrwać.

In [ ]:
pred_transformer = sentiment_pipeline(x_test_texts, batch_size=100, truncation=True)

In [ ]:
pred_trans_mapped = [map_label[item["label"]] for item in pred_transformer]

Obliczenie accuracy dla transformera

In [ ]:
transformer_acc = accuracy_score(y_test, pred_trans_mapped)

In [ ]:
# Comparison results
print(f"RNN Accuracy: {rnn_acc:.4f}")
print(f"LSTM Accuracy: {lstm_acc:.4f}")
print(f"Transformer Accuracy: {transformer_acc:.4f}")


### **Zadanie na zajęciach 7**  (EXTRA)

Wykonaj predykcję sentymentu dla 1000 losowych próbek ze zbioru IMDb, wykorzystując modele:
SimpleRNN, LSTM oraz transformer (`sentiment_pipeline`).

Następnie oblicz metryki:
accuracy, precision oraz recall dla każdego modelu i porównaj wyniki.

Cel:
- porównanie skuteczności różnych podejść do analizy tekstu

## Generacja tekstu

W tej części zajmiemy się generacją tekstu z wykorzystaniem modeli sekwencyjnych.

Uwaga:
- należy dodać plik `wonderland.txt` z folderu Lab4 (repozytorium) do środowiska Google Colab przed uruchomieniem kodu

Ładowanie danych

In [ ]:
filename = "wonderland.txt"
raw_text = open(filename, 'r', encoding='utf-8').read()
raw_text = raw_text.lower()

In [ ]:
# Stworzenie mapy znaków do int
chars = sorted(list(set(raw_text)))
char_to_int = dict((c, i) for i, c in enumerate(chars))

In [ ]:
n_chars = len(raw_text)
n_vocab = len(chars)
print("Total Characters: ", n_chars)
print("Total Vocab: ", n_vocab)

### Przygotowanie danych wejściowych

Aby trenować model do generacji tekstu, należy przygotować dane wejściowe w odpowiedniej formie.

- podział tekstu na pary (X, y):
  - X – fragment tekstu (wejście)
  - y – kolejny znak lub słowo (cel – kontynuacja)

- zamiana znaków lub słów na liczby (indeksy), aby model mógł je przetwarzać

In [ ]:
seq_length = 100
x_enc = []
y_enc = []
for i in range(0, n_chars - seq_length, 1):
    seq_in = raw_text[i:i + seq_length]
    seq_out = raw_text[i + seq_length]
    x_enc.append([char_to_int[char] for char in seq_in])
    y_enc.append(char_to_int[seq_out])
n_patterns = len(x_enc)
print("Total Patterns: ", n_patterns)

In [ ]:
x_enc[3][:5]

In [ ]:
y_enc[3]

In [ ]:
# reshape X do sieci [samples, time steps, features]
X = np.reshape(x_enc, (n_patterns, seq_length, 1))
# Normalizacja danych
X = X / float(n_vocab)
# kodowanie targetu
y = to_categorical(y_enc)

In [ ]:
# Definicja modelu
model = Sequential([
    LSTM(128, input_shape=(seq_length, 1)),
    Dropout(0.1),
    Dense(y.shape[1], activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam')

In [ ]:
# stworzenie mapy int to char - odwrócenie działania modelu
int_to_char = dict((i, c) for i, c in enumerate(chars))

In [ ]:
# definicja checkpointu modelu
filepath="weights-improvement-{epoch:02d}-{loss:.4f}.weights.h5"
checkpoint = ModelCheckpoint(filepath, monitor='loss', verbose=1, save_best_only=True, mode='min', save_weights_only=True)
callbacks_list = [checkpoint]

**TIP** Przy wielokrotnym trenowaniu modelu można wyłączyć tworzenie callbacku, żeby nie powielać zapisywania modeli.

In [ ]:
model.fit(X, y, epochs=5, batch_size=128, callbacks=callbacks_list)

In [ ]:
# generacja tekstu z losowym początkiem
start = np.random.randint(0, len(x_enc)-1)
pattern = x_enc[start]
start_pattern = copy.copy(pattern)
print("Seed:")
print("\"", ''.join([int_to_char[value] for value in pattern]), "\"")

In [ ]:
# generacja tekstu przez n iteracji
n = 25
generated_text = []

for i in range(n):
    x = np.reshape(pattern, (1, len(pattern), 1))
    # normalizacja
    x = x / float(n_vocab)
    prediction = model.predict(x, verbose=0)
    # Wybranie następnego znaku zgodnie z największym prawdopodobieństwem zwróconym przez model
    index = np.argmax(prediction)
    # konwersja predykcji do znaku
    result = int_to_char[index]
    pattern.append(index)
    generated_text.append(result)
    # aktualizacja danych wejściowych o wygenerowany znak - podobnie jak w predykcji dla szeregów czasowych
    pattern = pattern[1:len(pattern)]

Wygenerowany text

In [ ]:
print(("").join([item for item in generated_text]))

Połączony wygenerowany tekst z wzorem początkowym

In [ ]:
start_pattern_text = [int_to_char[p] for p in start_pattern]
print(("").join([item for item in [*start_pattern_text, *generated_text]]))

### **Zadanie na zajęciach 8**

Spróbuj poprawić jakość generowanego tekstu, eksperymentując z modelem i danymi wejściowymi.

Możliwe kierunki zmian:
- zwiększenie złożoności modelu LSTM (więcej warstw, **dropout** dla sieci rekurencyjnych)
- użycie parametru `return_sequences` w LSTM
- transformacja danych wejściowych do one-hot-embeddingu (metoda `to_categorical`)
- dodanie warstwy Embedding na wejściu
- zamiana LSTM na GRU (https://www.tensorflow.org/api_docs/python/tf/keras/layers/GRU)
- modyfikacja parametrów treningu (batch size, **liczba epok**)
- zmiana długości sekwencji wejściowej
- zmiana liczby generowanych znaków
- preprocessing tekstu (usunięcie zbędnych znaków)
- zmiana strategii generowania:
  - zamiast wyboru znaku o najwyższym prawdopodobieństwie
  - zastosowanie losowania ważonego (np. z top-n najbardziej prawdopodobnych)

Na koniec:
- porównaj jakościowo wygenerowane teksty
- sprawdź i podaj liczbę parametrów trenowalnych w modelu

Cel:
- zrozumienie wpływu architektury i parametrów na jakość generacji tekstu

### **Zadanie na zajęciach ($$$)**

Zaimplementuj własną klasę komórki LSTM zgodnie z oryginalną publikacją:
https://doi.org/10.1162/neco.1997.9.8.1735

Możesz skorzystać z materiałów pomocniczych:
- https://www.geeksforgeeks.org/deep-learning-introduction-to-long-short-term-memory/
- https://colah.github.io/posts/2015-08-Understanding-LSTMs/

Wymagania:
- zaimplementuj tylko propagację w przód (forward pass)
- odwzoruj mechanizm bramek:
  - input gate
  - forget gate
  - output gate
- uwzględnij aktualizację stanu komórki (cell state) i stanu ukrytego (hidden state)

Test:
- wygeneruj sztuczne dane wejściowe oraz wagi
- uruchom forward pass
- sprawdź poprawność wymiarów i stabilność działania

Cel:
- zrozumienie wewnętrznej mechaniki działania komórki LSTM